# RCPCHgrowth Quickstart

This notebook is for researchers wishing to  use the RCPCHGrowth calculations without using the RCPCH digital growth charts API. This explainer will guide on how to:

1. Inspect your environment & package versions.
2. Calculate a single z-score (SDS) & centile for a child's measurement.
3. Process a small batch of measurements from a pandas DataFrame.
4. Plot a simple growth trajectory against centile bands (weight example).
5. (Bonus) Export results.

It also demonstrates good reproducibility practice: fixed versions, explicit unit notes, and data schema guidance.

> IMPORTANT: Do not place real identifiable patient data into a public clone of this repository. Keep research data local & de‑identified.


In [7]:
# Environment & versions (works whether installed via pip or run from cloned repo)
import sys, platform, pathlib
import pandas as pd

try:
    import rcpchgrowth  # normal case: installed package
except ImportError:
    # Fallback: we're likely in a cloned repo and haven't pip-installed yet.
    repo_root = pathlib.Path().resolve().parent  # notebooks/ -> repo root parent
    if (repo_root / 'setup.py').exists():
        sys.path.insert(0, str(repo_root))
        import rcpchgrowth
    else:
        raise  # re-raise if truly not available

print({
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'rcpchgrowth': getattr(rcpchgrowth, '__version__', 'unknown'),
    'rcpchgrowth_module_path': getattr(rcpchgrowth, '__file__', 'n/a'),
    'pandas': pd.__version__,
})

{'python': '3.12.0', 'platform': 'macOS-15.6-arm64-arm-64bit', 'rcpchgrowth': 'unknown', 'rcpchgrowth_module_path': '/Users/eatyourpeas/Development/RCPCH repositories/growth charts/rcpchgrowth-python/rcpchgrowth/__init__.py', 'pandas': '2.3.1'}


## 1. Single measurement example
We will compute a weight SDS (z-score) and centile for a fictitious child.

Assumptions / Inputs (mandatory):
- `sex`: 'male' or 'female'
- `birth_date` - supplied as a python Date
- `observation_date` - the date the child was measured: supplied as a python Date
- `observation_value` - the measurement: note the units are standard - only cm or kg or kg/m2 are accepted.
- `measurement_method` - the type of measurement performed. Options include: `['height', 'weight' , 'bmi', 'ofc']`. (ofc = 'occipto-frontal circumference', bmi='body mass index'). Note this must be lower case.

Optional Inputs
- `gestation_weeks`: integer. Defaults to 40 if not supplied
- `gestation_days`: integer. Defaults to 0 if not supplied
- `reference`: this defaults to UK-WHO. Other options include: `['trisomy-21', 'trisomy-21-aap', 'turner-syndrome', 'cdc', 'who']`. For more information on the dataset please see our (documentation)['https://growth.rcpch.ac.uk']

The Measurement class performs all the calculations and returns the results as a python dictionary. It accepts the parameters above and returns the class object. The results are accessed through the `measurement` class attribute.


In [9]:
from datetime import date
from rcpchgrowth import Measurement

sex = 'female'
dob = date(2022, 6, 15)
md  = date(2024, 2, 1)
weight_kg = 12.3

measurement = Measurement(sex=sex, birth_date=dob, measurement_method='weight', observation_date=md, observation_value=weight_kg, reference='uk-who', gestation_weeks=40, gestation_days=0).measurement

# Extracting the results from the measurement dictionary

# Calculated ages
chronological_age_decimal_years = measurement['measurement_dates']["chronological_decimal_age"]
corrected_age_decimal_years = measurement['measurement_dates']["corrected_decimal_age"]
chronological_calendar_age = measurement['measurement_dates']["chronological_calendar_age"] # returns age as readable text in years, months, weeks and days
corrected_calendar_age = measurement['measurement_dates']["corrected_calendar_age"] # returns age as readable text in years, months, weeks and days
# This returns corrected gestational age in weeks if the baby was premature and is not yet term.
corrected_gestational_age = measurement['measurement_dates']["corrected_gestational_age"]["corrected_gestation_weeks"]
corrected_gestational_age = measurement['measurement_dates']["corrected_gestational_age"]["corrected_gestation_days"]

# calculated SDS and centiles
corrected_weight_sds = measurement["measurement_calculated_values"]["corrected_sds"]
corrected_weight_centile = measurement["measurement_calculated_values"]["corrected_centile"]
chronological_weight_sds = measurement["measurement_calculated_values"]["chronological_sds"]
chronological_weight_centile = measurement["measurement_calculated_values"]["chronological_centile"]

print(f"Age (decimal years): {chronological_age_decimal_years:.3f}")
print(f"Weight: {weight_kg} kg | SDS: {corrected_weight_sds:.2f} | Centile: {corrected_weight_centile:.1f}")

Age (decimal years): 1.632
Weight: 12.3 kg | SDS: 1.21 | Centile: 88.7


## 2. Batch processing with a DataFrame
For research datasets you typically have many rows. In this example we have created a miniature DataFrame to show vectorised processing.

Expected columns (example):
sex = 'female'
dob = date(2022, 6, 15)
md  = date(2024, 2, 1)
weight_kg = 12.3

We'll compute age, retrieve growth data per row, and create SDS & centile columns.


In [ ]:
import pandas as pd
from datetime import date
from rcpchgrowth import Measurement

rows = [
    {'id': 'A', 'sex': 'F', 'dob': date(2022, 6, 15), 'measurement_date': date(2024, 2, 1), 'weight_kg': 12.3},
    {'id': 'B', 'sex': 'M', 'dob': date(2021,11,10), 'measurement_date': date(2024, 2, 1), 'weight_kg': 15.0},
    {'id': 'C', 'sex': 'F', 'dob': date(2023, 4,  5), 'measurement_date': date(2024, 2, 1), 'weight_kg':  8.4},
]

df = pd.DataFrame(rows)

def compute_row(row):
    measurement = Measurement(birth_date=row.dob, measurement_method='weight', observation_date=row.measurement_date, observation_value=row.weight_kg, reference='uk-who', gestation_weeks=40, gestation_days=0).measurement
    age = measurement['measurement_dates']["chronological_decimal_age"]
    corrected_weight_sds = measurement["measurement_calculated_values"]["corrected_sds"]
    corrected_weight_centile = measurement["measurement_calculated_values"]["corrected_centile"]
    return pd.Series({'age_decimal_years': age, 'weight_sds': corrected_weight_sds, 'weight_centile': corrected_weight_centile})

calc = df.apply(compute_row, axis=1)
df = pd.concat([df, calc], axis=1)
df

## 3. Simple trajectory plot
We'll plot weight SDS over age for the batch. For richer visualisations you might overlay centile bands using `create_chart` or export to a separate plotting library.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,3))
plt.axhline(0, color='lightgray', lw=1)
plt.scatter(df['age_decimal_years'], df['weight_sds'], c=['tab:blue','tab:orange','tab:green'])
for _, r in df.iterrows():
    plt.text(r['age_decimal_years']+0.01, r['weight_sds'], r['id'])
plt.xlabel('Age (decimal years)')
plt.ylabel('Weight SDS')
plt.title('Weight SDS trajectory (example)')
plt.tight_layout()
plt.show()

## 4. Export results
You can now export the enriched DataFrame (with SDS & centiles) to CSV for downstream stats or modelling.


In [ ]:
# Export (disabled by default) - uncomment to write
# df.to_csv('example_results.csv', index=False)
print('DataFrame ready; uncomment export line to save to CSV.')